This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [ ]:
!pip install torch torchvision --upgrade -q

## Image classification

### Introduction to ConvNets

In [ ]:
import torch
from torch import nn

# PyTorch: Conv2D stacks become nn.Conv2d in NCHW layout (channels first), so the
# input is (1, 28, 28) rather than Keras's NHWC (28, 28, 1). We drop the final
# softmax because nn.CrossEntropyLoss expects raw logits, and global average
# pooling is done in forward() with x.mean over the spatial dims.
class ConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 64, kernel_size=3)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.classifier = nn.Linear(256, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.relu(self.conv2(x))
        x = self.pool(x)
        x = torch.relu(self.conv3(x))
        x = x.mean(dim=(2, 3))
        return self.classifier(x)

model = ConvNet()

In [ ]:
# PyTorch: there is no model.summary(); printing the module lists its layers.
print(model)

In [ ]:
# PyTorch: load MNIST via torchvision and expose raw uint8 NumPy arrays. We reshape
# to NCHW (N, 1, 28, 28) instead of Keras's NHWC, and replace compile()/fit() with
# an explicit training loop using nn.CrossEntropyLoss (softmax + sparse CE).
from torchvision.datasets import MNIST

mnist_train = MNIST(root="./data", train=True, download=True)
mnist_test = MNIST(root="./data", train=False, download=True)
train_images, train_labels = mnist_train.data.numpy(), mnist_train.targets.numpy()
test_images, test_labels = mnist_test.data.numpy(), mnist_test.targets.numpy()
train_images = train_images.reshape((60000, 1, 28, 28))
train_images = train_images.astype("float32") / 255
test_images = test_images.reshape((10000, 1, 28, 28))
test_images = test_images.astype("float32") / 255

optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()

x_train = torch.from_numpy(train_images)
y_train = torch.from_numpy(train_labels).long()
batch_size = 64
for epoch in range(5):
    permutation = torch.randperm(len(x_train))
    for i in range(0, len(x_train), batch_size):
        idx = permutation[i : i + batch_size]
        inputs, targets = x_train[idx], y_train[idx]
        predictions = model(inputs)
        loss = loss_fn(predictions, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch}: loss {loss.item():.4f}")

In [ ]:
# PyTorch: model.evaluate() becomes a manual no-grad accuracy pass.
model.eval()
with torch.no_grad():
    predicted_labels = model(torch.from_numpy(test_images)).argmax(dim=1).numpy()
test_acc = (predicted_labels == test_labels).mean()
print(f"Test accuracy: {test_acc:.3f}")

#### The convolution operation

##### Understanding border effects and padding

##### Understanding convolution strides

#### The max-pooling operation

In [ ]:
# PyTorch: same convnet without max-pooling, as an nn.Module (NCHW).
class ConvNetNoMaxPool(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 64, kernel_size=3)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3)
        self.classifier = nn.Linear(256, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.relu(self.conv3(x))
        x = x.mean(dim=(2, 3))
        return self.classifier(x)

model_no_max_pool = ConvNetNoMaxPool()

In [ ]:
print(model_no_max_pool)

### Training a ConvNet from scratch on a small dataset

#### The relevance of deep learning for small-data problems

#### Downloading the data

In [0]:
import kagglehub

kagglehub.login()

In [0]:
download_path = kagglehub.competition_download("dogs-vs-cats")

In [0]:
import zipfile

with zipfile.ZipFile(download_path + "/train.zip", "r") as zip_ref:
    zip_ref.extractall(".")

In [0]:
import os, shutil, pathlib

original_dir = pathlib.Path("train")
new_base_dir = pathlib.Path("dogs_vs_cats_small")

def make_subset(subset_name, start_index, end_index):
    for category in ("cat", "dog"):
        dir = new_base_dir / subset_name / category
        os.makedirs(dir)
        fnames = [f"{category}.{i}.jpg" for i in range(start_index, end_index)]
        for fname in fnames:
            shutil.copyfile(src=original_dir / fname, dst=dir / fname)

make_subset("train", start_index=0, end_index=1000)
make_subset("validation", start_index=1000, end_index=1500)
make_subset("test", start_index=1500, end_index=2500)

#### Building your model

In [ ]:
import torch
from torch import nn

# PyTorch: the dogs-vs-cats convnet as an nn.Module in NCHW. Rescaling(1/255) is
# folded into the data pipeline (ToTensor already scales to [0,1]). We drop the
# final sigmoid because nn.BCEWithLogitsLoss expects raw logits, and use a single
# output unit for binary classification.
class DogsVsCatsConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3)
        self.conv5 = nn.Conv2d(256, 512, kernel_size=3)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.classifier = nn.Linear(512, 1)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))
        x = self.pool(torch.relu(self.conv4(x)))
        x = torch.relu(self.conv5(x))
        x = x.mean(dim=(2, 3))
        return self.classifier(x)

model = DogsVsCatsConvNet()

In [ ]:
print(model)

In [ ]:
# PyTorch: compile() becomes an explicit optimizer + loss. BCEWithLogitsLoss is
# the binary-crossentropy equivalent and expects raw logits (no sigmoid).
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.BCEWithLogitsLoss()

#### Data preprocessing

In [ ]:
# PyTorch: image_dataset_from_directory becomes torchvision.datasets.ImageFolder
# + DataLoader. Resize matches image_size, ToTensor produces NCHW float tensors in
# [0, 1] (covering Keras's Rescaling 1/255). ImageFolder sorts classes
# alphabetically, so cat=0 / dog=1, matching Keras's label assignment.
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder

batch_size = 64
image_size = (180, 180)
base_transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
])
train_dataset = ImageFolder(new_base_dir / "train", transform=base_transform)
validation_dataset = ImageFolder(new_base_dir / "validation", transform=base_transform)
test_dataset = ImageFolder(new_base_dir / "test", transform=base_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

##### Understanding TensorFlow Dataset objects

In [ ]:
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

# PyTorch: tf.data.Dataset.from_tensor_slices becomes a TensorDataset wrapping the
# sample tensor; iterating yields one row at a time.
random_numbers = np.random.normal(size=(1000, 16))
dataset = TensorDataset(torch.from_numpy(random_numbers))

In [ ]:
for i, (element,) in enumerate(dataset):
    print(element.shape)
    if i >= 2:
        break

In [ ]:
# PyTorch: dataset.batch(32) becomes a DataLoader with batch_size=32.
batched_dataset = DataLoader(dataset, batch_size=32)
for i, (element,) in enumerate(batched_dataset):
    print(element.shape)
    if i >= 2:
        break

In [ ]:
# PyTorch: dataset.map(lambda x: reshape) becomes reshaping each element as we
# iterate (no parallel map abstraction needed here).
for i, (element,) in enumerate(dataset):
    reshaped = element.reshape(4, 4)
    print(reshaped.shape)
    if i >= 2:
        break

##### Fitting the model

In [ ]:
# PyTorch: DataLoader batches are NCHW tensors; labels are a 1-D int tensor.
for data_batch, labels_batch in train_loader:
    print("data batch shape:", data_batch.shape)
    print("labels batch shape:", labels_batch.shape)
    break

In [ ]:
# PyTorch: model.fit() + ModelCheckpoint(save_best_only) become an explicit
# training loop. After each epoch we run a no-grad validation pass and save the
# state_dict whenever val_loss improves. We collect per-epoch metrics into a dict
# shaped like Keras's history.history so the plotting cell still works.
device = "cuda" if torch.cuda.is_available() else "cpu"

def run_epoch(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, correct, total = 0.0, 0, 0
    torch.set_grad_enabled(is_train)
    for images, labels in loader:
        images = images.to(device)
        targets = labels.float().unsqueeze(1).to(device)
        logits = model(images)
        loss = loss_fn(logits, targets)
        if is_train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == targets).sum().item()
        total += images.size(0)
    torch.set_grad_enabled(True)
    return total_loss / total, correct / total

def train_model(model, train_loader, validation_loader, optimizer, loss_fn,
                epochs, filepath):
    model.to(device)
    history = {"accuracy": [], "loss": [], "val_accuracy": [], "val_loss": []}
    best_val_loss = float("inf")
    for epoch in range(epochs):
        tr_loss, tr_acc = run_epoch(model, train_loader, loss_fn, optimizer)
        val_loss, val_acc = run_epoch(model, validation_loader, loss_fn)
        history["loss"].append(tr_loss)
        history["accuracy"].append(tr_acc)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), filepath)
        print(f"Epoch {epoch + 1}/{epochs} - loss {tr_loss:.4f} acc {tr_acc:.3f} "
              f"- val_loss {val_loss:.4f} val_acc {val_acc:.3f}")
    return history

history = train_model(
    model, train_loader, validation_loader, optimizer, loss_fn,
    epochs=50, filepath="convnet_from_scratch.pt",
)

In [0]:
import matplotlib.pyplot as plt

accuracy = history.history["accuracy"]
val_accuracy = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs = range(1, len(accuracy) + 1)

plt.plot(epochs, accuracy, "r--", label="Training accuracy")
plt.plot(epochs, val_accuracy, "b", label="Validation accuracy")
plt.title("Training and validation accuracy")
plt.legend()
plt.figure()

plt.plot(epochs, loss, "r--", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.legend()
plt.show()

In [ ]:
# PyTorch: load_model becomes re-instantiating the module and loading the
# saved state_dict; evaluate becomes a no-grad pass via run_epoch.
test_model = DogsVsCatsConvNet().to(device)
test_model.load_state_dict(torch.load("convnet_from_scratch.pt"))
test_loss, test_acc = run_epoch(test_model, test_loader, loss_fn)
print(f"Test accuracy: {test_acc:.3f}")

#### Using data augmentation

In [ ]:
# PyTorch: Keras RandomFlip/RandomRotation/RandomZoom become torchvision
# transforms in the train pipeline. RandomResizedCrop approximates RandomZoom.
# We build a separate train transform and rebuild the train dataset/loader with it.
augment_transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=36),
    transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0)),
    transforms.ToTensor(),
])
augmented_train_dataset = ImageFolder(new_base_dir / "train", transform=augment_transform)
augmented_train_loader = DataLoader(augmented_train_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
# PyTorch: visualize augmentations by applying the augment transform to one image
# and converting the CHW float tensor back to an HWC uint8 array for imshow.
from PIL import Image

plt.figure(figsize=(10, 10))
sample_path, _ = augmented_train_dataset.samples[0]
sample_image = Image.open(sample_path).convert("RGB")
for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    augmented_image = augment_transform(sample_image)
    augmented_image = (augmented_image.permute(1, 2, 0).numpy() * 255).astype("uint8")
    plt.imshow(augmented_image)
    plt.axis("off")

In [ ]:
# PyTorch: same convnet as before but with a Dropout before the classifier; the
# augmentation lives in the data pipeline (cell above), not as model layers.
class DogsVsCatsConvNetDropout(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3)
        self.conv5 = nn.Conv2d(256, 512, kernel_size=3)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.dropout = nn.Dropout(0.25)
        self.classifier = nn.Linear(512, 1)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))
        x = self.pool(torch.relu(self.conv4(x)))
        x = torch.relu(self.conv5(x))
        x = x.mean(dim=(2, 3))
        x = self.dropout(x)
        return self.classifier(x)

model = DogsVsCatsConvNetDropout()
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.BCEWithLogitsLoss()

In [ ]:
history = train_model(
    model, augmented_train_loader, validation_loader, optimizer, loss_fn,
    epochs=100, filepath="convnet_from_scratch_with_augmentation.pt",
)

In [ ]:
test_model = DogsVsCatsConvNetDropout().to(device)
test_model.load_state_dict(torch.load("convnet_from_scratch_with_augmentation.pt"))
test_loss, test_acc = run_epoch(test_model, test_loader, loss_fn)
print(f"Test accuracy: {test_acc:.3f}")

### Using a pretrained model

#### Feature extraction with a pretrained model

In [ ]:
# PyTorch: keras_hub's Xception ImageNet backbone has no direct PyTorch weights, so
# we substitute a torchvision ResNet-50 pretrained on ImageNet as the conv base.
# We strip its classifier head (fc) so it outputs a feature map / pooled features.
from torchvision.models import resnet50, ResNet50_Weights

weights = ResNet50_Weights.IMAGENET1K_V2
conv_base = resnet50(weights=weights)
# PyTorch: keep everything up to (but not including) the final avgpool+fc, so the
# backbone emits the (2048, H, W) feature map like Keras's Backbone.
conv_base = nn.Sequential(*list(conv_base.children())[:-2])

In [ ]:
# PyTorch: keras_hub's ImageConverter becomes the transform bundled with the
# pretrained weights (resize + normalize with ImageNet stats), resized to 180x180.
preprocessor = transforms.Compose([
    transforms.Resize((180, 180)),
    transforms.ToTensor(),
    transforms.Normalize(mean=weights.transforms().mean,
                         std=weights.transforms().std),
])
# PyTorch: rebuild the ImageFolder datasets/loaders with this preprocessing so the
# feature-extraction loop below feeds the backbone correctly.
prep_train = DataLoader(ImageFolder(new_base_dir / "train", transform=preprocessor),
                        batch_size=batch_size)
prep_val = DataLoader(ImageFolder(new_base_dir / "validation", transform=preprocessor),
                      batch_size=batch_size)
prep_test = DataLoader(ImageFolder(new_base_dir / "test", transform=preprocessor),
                       batch_size=batch_size)

##### Fast feature extraction without data augmentation

In [ ]:
# PyTorch: run the frozen backbone over each loader and collect features/labels.
# Images are already preprocessed by the DataLoader transform, so we just forward.
conv_base.to(device).eval()

def get_features_and_labels(loader):
    all_features = []
    all_labels = []
    with torch.no_grad():
        for images, labels in loader:
            features = conv_base(images.to(device))
            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())
    return np.concatenate(all_features), np.concatenate(all_labels)

train_features, train_labels = get_features_and_labels(prep_train)
val_features, val_labels = get_features_and_labels(prep_val)
test_features, test_labels = get_features_and_labels(prep_test)

In [ ]:
train_features.shape

In [ ]:
# PyTorch: the small classifier head over the (2048, H, W) feature maps. Global
# average pooling is x.mean over spatial dims; final sigmoid is dropped in favor of
# BCEWithLogitsLoss. We train it with an explicit loop over the cached features.
from torch.utils.data import TensorDataset

class FeatureClassifier(nn.Module):
    def __init__(self, in_channels=2048):
        super().__init__()
        self.fc1 = nn.Linear(in_channels, 256)
        self.dropout = nn.Dropout(0.25)
        self.fc2 = nn.Linear(256, 1)

    def forward(self, x):
        x = x.mean(dim=(2, 3))
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

model = FeatureClassifier(in_channels=train_features.shape[1])
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.BCEWithLogitsLoss()

feat_train_loader = DataLoader(
    TensorDataset(torch.from_numpy(train_features), torch.from_numpy(train_labels)),
    batch_size=batch_size, shuffle=True,
)
feat_val_loader = DataLoader(
    TensorDataset(torch.from_numpy(val_features), torch.from_numpy(val_labels)),
    batch_size=batch_size,
)
history = train_model(
    model, feat_train_loader, feat_val_loader, optimizer, loss_fn,
    epochs=10, filepath="feature_extraction.pt",
)

In [0]:
import matplotlib.pyplot as plt

acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs = range(1, len(acc) + 1)
plt.plot(epochs, acc, "r--", label="Training accuracy")
plt.plot(epochs, val_acc, "b", label="Validation accuracy")
plt.title("Training and validation accuracy")
plt.legend()
plt.figure()
plt.plot(epochs, loss, "r--", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.legend()
plt.show()

In [ ]:
test_model = FeatureClassifier(in_channels=test_features.shape[1]).to(device)
test_model.load_state_dict(torch.load("feature_extraction.pt"))
feat_test_loader = DataLoader(
    TensorDataset(torch.from_numpy(test_features), torch.from_numpy(test_labels)),
    batch_size=batch_size,
)
test_loss, test_acc = run_epoch(test_model, feat_test_loader, loss_fn)
print(f"Test accuracy: {test_acc:.3f}")

##### Feature extraction together with data augmentation

In [ ]:
# PyTorch: reload the ResNet-50 conv base and freeze it (requires_grad=False) so
# only the new head trains, mirroring keras_hub's trainable=False.
weights = ResNet50_Weights.IMAGENET1K_V2
conv_base = resnet50(weights=weights)
conv_base = nn.Sequential(*list(conv_base.children())[:-2])
for param in conv_base.parameters():
    param.requires_grad = False

In [ ]:
# PyTorch: there is no .trainable flag; we toggle requires_grad and count the
# parameter tensors that are currently trainable.
for param in conv_base.parameters():
    param.requires_grad = True
len([p for p in conv_base.parameters() if p.requires_grad])

In [ ]:
for param in conv_base.parameters():
    param.requires_grad = False
len([p for p in conv_base.parameters() if p.requires_grad])

In [ ]:
# PyTorch: an end-to-end model that runs the (frozen) backbone then the head.
# Augmentation/preprocessing lives in the data pipeline; preprocessing for the
# backbone is the ImageNet normalize applied via the augmented loader below.
class TransferModel(nn.Module):
    def __init__(self, conv_base, in_channels=2048):
        super().__init__()
        self.conv_base = conv_base
        self.fc1 = nn.Linear(in_channels, 256)
        self.dropout = nn.Dropout(0.25)
        self.fc2 = nn.Linear(256, 1)

    def forward(self, x):
        x = self.conv_base(x)
        x = x.mean(dim=(2, 3))
        x = self.fc1(x)
        x = self.dropout(x)
        return self.fc2(x)

model = TransferModel(conv_base)
# PyTorch: optimize only the parameters that require grad (the head, since the
# backbone is frozen here).
optimizer = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad]
)
loss_fn = nn.BCEWithLogitsLoss()

In [ ]:
# PyTorch: build an augmented train loader that also applies ImageNet normalization
# (the backbone expects normalized inputs), then run the training loop.
transfer_augment = transforms.Compose([
    transforms.Resize(image_size),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=36),
    transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=weights.transforms().mean,
                         std=weights.transforms().std),
])
transfer_train_loader = DataLoader(
    ImageFolder(new_base_dir / "train", transform=transfer_augment),
    batch_size=batch_size, shuffle=True,
)
history = train_model(
    model, transfer_train_loader, prep_val, optimizer, loss_fn,
    epochs=30, filepath="feature_extraction_with_data_augmentation.pt",
)

In [ ]:
test_model = TransferModel(nn.Sequential(
    *list(resnet50(weights=weights).children())[:-2]
)).to(device)
test_model.load_state_dict(torch.load("feature_extraction_with_data_augmentation.pt"))
test_loss, test_acc = run_epoch(test_model, prep_test, loss_fn)
print(f"Test accuracy: {test_acc:.3f}")

#### Fine-tuning a pretrained model

In [ ]:
# PyTorch: fine-tuning unfreezes the backbone and retrains the whole model with a
# small learning rate. We re-enable grads on all params and build a fresh Adam(1e-5).
for param in model.parameters():
    param.requires_grad = True
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
loss_fn = nn.BCEWithLogitsLoss()
history = train_model(
    model, transfer_train_loader, prep_val, optimizer, loss_fn,
    epochs=30, filepath="fine_tuning.pt",
)

In [ ]:
model = TransferModel(nn.Sequential(
    *list(resnet50(weights=weights).children())[:-2]
)).to(device)
model.load_state_dict(torch.load("fine_tuning.pt"))
test_loss, test_acc = run_epoch(model, prep_test, loss_fn)
print(f"Test accuracy: {test_acc:.3f}")